# APM_50441_EP - Machine Learning 2 - Lab 2: Inside the methods, and the tidymodels workflow

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/APM-50441-EP/APM_50441_EP-2026-2027/blob/main/PC2/APM_50441_EP_Lab_2_simplified_with_solution.ipynb) or run it on your own computer (see *Working environment* below).

In [ ]:
# On Colab (R runtime) only: installs the packages of this lab (once per session, a few minutes).
# On your own computer, use PC/env/install_packages.R instead (see Working environment).
if (nzchar(Sys.getenv("COLAB_RELEASE_TAG"))) {
  ubuntu <- sub("^VERSION_CODENAME=", "", grep("^VERSION_CODENAME=", readLines("/etc/os-release"), value = TRUE))
  options(repos = c(CRAN = paste0("https://packagemanager.posit.co/cran/__linux__/", ubuntu, "/latest")))  # binaries for this Ubuntu
  source("https://raw.githubusercontent.com/APM-50441-EP/APM_50441_EP-2026-2027/main/env/install_packages.R")
  install_lab("lab2")
}

This lab illustrates the second lecture (*A Review of the Methods Seen So Far*). Its goals are:

- to compare classifiers with the *truth*: on a simulated problem, we know the conditional probability $\eta(x) = \mathbb{P}(Y = \text{Class1} | X = x)$, hence the Bayes classifier and its risk $\mathcal{R}^\star$, and we can measure the *excess risk* of each method,
- to see the two points of view of the lecture at work: the probabilistic one (plug-in classifiers) and the optimization one (empirical risk minimization of a convex surrogate),
- to measure the bias–variance trade-off of a local method,
- to learn the `tidymodels` workflow (recipes, workflows, tuning, workflow sets, metrics) on the TwoClass data of the lecture.

The optional *To go further* part, after the core, can be skipped at first.

We use `R` with `tidyverse` and `tidymodels`.

**Working environment.** Colab is the simplest option: open the notebook version of this lab (`APM_50441_EP_Lab_2.ipynb`, *Open in Colab* link) with the R runtime, and run its first cell, which installs the packages. To work on your own computer, download the lab folder (the `.qmd` file and its files) or clone the course repository, open the `.qmd` file in RStudio or Positron (or VS Code with the R extension), and run it chunk by chunk or render it with Quarto. The packages are installed once with the script `PC/env/install_packages.R` (`source("install_packages.R")`, then `install_lab("lab2")`). The data are read from the local files when they are present (otherwise they are downloaded).

**Use of AI.** AI assistants are allowed, but you must be able to explain every line you use, and you should answer the *predict before you run* questions before running the code: the suggested code may not be correct!

In [ ]:
library(tidyverse)
library(tidymodels)
library(discrim)      # LDA, QDA and naive Bayes in parsnip
library(patchwork)
library(mvtnorm)
tidymodels_prefer()
theme_set(theme_light())
if (!file.exists("data/twoClassData.csv")) {  # downloaded only if there is no local copy
  dir.create("data", showWarnings = FALSE)
  download.file("https://raw.githubusercontent.com/APM-50441-EP/APM_50441_EP-2026-2027/main/PC2/data/twoClassData.csv",
                "data/twoClassData.csv")
}

## 1. The problem and its Bayes classifier

We simulate a two-class problem in which each class is Gaussian, with different covariances: $X | Y = \text{Class1} \sim \mathcal{N}(\mu_1, \Sigma_1)$ and $X | Y = \text{Class2} \sim \mathcal{N}(\mu_2, \Sigma_2)$, with equal priors. As the model is known, $\eta(x)$ can be computed with Bayes' formula. The following helpers are given:

- `simulate(n, noise = 0)` draws $n$ observations (with `noise` additional useless features),
- `eta(data)` computes $\eta(x)$ for each row,
- `excess_risk(pred_class, test)` computes the excess risk $\mathcal{R}(\hat f) - \mathcal{R}^\star = \mathbb{E}\left[\mathbf{1}_{\hat f(X) \neq f^\star(X)} |2\eta(X) - 1|\right]$ on a test set where `eta` is known (this formula has a much smaller variance than the difference of two empirical risks),
- `decision_map(fitted, data)` plots the estimated probability of `Class1`, the decision boundary of a fitted model (solid) and the Bayes boundary (dashed).

In [ ]:
mu1 <- c(0, 0); Sigma1 <- diag(2)
mu2 <- c(1.5, 1); Sigma2 <- matrix(c(2, 0.9, 0.9, 0.6), 2)

simulate <- function(n, noise = 0) {
  y <- rbinom(n, 1, 0.5)
  x <- matrix(0, n, 2)
  x[y == 0, ] <- rmvnorm(sum(y == 0), mu1, Sigma1)
  x[y == 1, ] <- rmvnorm(sum(y == 1), mu2, Sigma2)
  data <- tibble(x1 = x[, 1], x2 = x[, 2])
  if (noise > 0) {
    data <- bind_cols(data, as_tibble(matrix(rnorm(n * noise), n, noise), .name_repair = ~ paste0("noise", seq_along(.x))))
  }
  data |> mutate(class = factor(if_else(y == 0, "Class1", "Class2"), levels = c("Class1", "Class2")))
}

eta <- function(data) {
  x <- cbind(data$x1, data$x2)
  d1 <- dmvnorm(x, mu1, Sigma1)
  d2 <- dmvnorm(x, mu2, Sigma2)
  d1 / (d1 + d2)
}

excess_risk <- function(pred_class, test) {
  bayes <- if_else(test$eta > 0.5, "Class1", "Class2")
  mean((as.character(pred_class) != bayes) * abs(2 * test$eta - 1))
}

grid <- expand_grid(x1 = seq(-3, 5, length.out = 120), x2 = seq(-3, 4, length.out = 120))
grid <- grid |> mutate(eta = eta(grid))

decision_map <- function(fitted, data, title = "") {
  pred <- grid |> bind_cols(predict(fitted, grid, type = "prob"))
  ggplot(pred, aes(x1, x2)) +
    geom_raster(aes(fill = .pred_Class1), alpha = 0.6) +
    geom_contour(aes(z = .pred_Class1), breaks = 0.5, color = "black") +
    geom_contour(aes(z = eta), breaks = 0.5, color = "black", linetype = "dashed") +
    geom_point(data = data, aes(color = class), size = 0.8) +
    scale_fill_gradient2(low = "#2a78d6", mid = "white", high = "#eb6834", midpoint = 0.5, limits = c(0, 1)) +
    scale_color_manual(values = c(Class1 = "#eb6834", Class2 = "#2a78d6")) +
    coord_equal(expand = FALSE) + labs(title = title, fill = "P(Class1)")
}

set.seed(42)
test <- simulate(20000)
test <- test |> mutate(eta = eta(test))
train <- simulate(300)

**Question** Plot `train` with the Bayes boundary $\eta(x) = 1/2$, and estimate the Bayes risk $\mathcal{R}^\star = \mathbb{E}[\min(\eta(X), 1 - \eta(X))]$ with `test`. *Predict before you run:* what is the shape of the Bayes boundary?

**Answer**

In [ ]:
ggplot(grid, aes(x1, x2)) +
  geom_raster(aes(fill = eta), alpha = 0.6) +
  geom_contour(aes(z = eta), breaks = 0.5, color = "black", linetype = "dashed") +
  geom_point(data = train, aes(color = class), size = 0.8) +
  scale_fill_gradient2(low = "#2a78d6", mid = "white", high = "#eb6834", midpoint = 0.5) +
  scale_color_manual(values = c(Class1 = "#eb6834", Class2 = "#2a78d6")) +
  coord_equal(expand = FALSE)
bayes_risk <- mean(pmin(test$eta, 1 - test$eta))
bayes_risk

**Conclusion** The two classes have different covariances, so the log-odds $\log\frac{\eta(x)}{1-\eta(x)}$ is a quadratic function of $x$ and the Bayes boundary is a hyperbola. The precision difference has eigenvalues approximately $-0.590$ and $5.257$, so the plot shows only part of the boundary. Even the best possible classifier makes errors: the Bayes risk is $\mathcal{R}^\star \approx 0.19$, as the classes overlap. All the risks below are compared with this value.

---

## 2. Plug-in classifiers and the estimation of $\eta$

In the probabilistic point of view, we estimate $\eta$ and plug the estimate in the Bayes rule: $\hat f(x) = \text{Class1}$ if $\hat\eta(x) > 1/2$. In `tidymodels`, a model is described by a *specification* (`parsnip`): its type, its mode and the *engine* (the R package) that fits it. All the specifications share the same `fit()` and `predict()` interface, and `predict(type = "prob")` gives the estimated probabilities.

In [ ]:
specifications <- tribble(
  ~method,                ~formula,                                                       ~spec,
  "logistic",             class ~ x1 + x2,                                                 logistic_reg(),
  "quadratic logistic",   class ~ x1 + x2 + I(x1^2) + I(x2^2) + I(x1 * x2),                logistic_reg(),
  "kNN (k = 25)",         class ~ x1 + x2,                                                 nearest_neighbor(neighbors = 25) |> set_mode("classification"),
  "LDA",                  class ~ x1 + x2,                                                 discrim_linear(),
  "QDA",                  class ~ x1 + x2,                                                 discrim_quad(),
  "Gaussian naive Bayes", class ~ x1 + x2,                                                 naive_Bayes() |> set_engine("naivebayes")
)
fit_method <- function(formula, spec, data) fit(workflow(formula, spec), data)

**Question** Fit the six methods on `train`, and plot their decision maps. Which methods can represent the Bayes boundary?

**Answer**

In [ ]:
fitted <- specifications |> mutate(fitted = map2(formula, spec, fit_method, data = train))
map2(fitted$fitted, fitted$method, decision_map, data = train) |> wrap_plots(ncol = 3, guides = "collect")

**Conclusion** The logistic regression and the LDA produce a linear boundary, and the Gaussian naive Bayes a quadratic boundary with axis-aligned covariances: none of them can represent the Bayes boundary exactly (they are *misspecified*). The quadratic logistic regression and the QDA have the right form: the QDA *is* the true model, and the log-odds of the quadratic logistic regression is a general quadratic function, which contains the true one. The $k$-NN can approximate any boundary, but its boundary is irregular: it pays its flexibility with variance.

---

PC2 (Exercise 1) shows that the excess risk of a plug-in classifier is controlled by the quality of $\hat\eta$:
$$\mathcal{R}(\hat f) - \mathcal{R}^\star \leq 2\, \mathbb{E}\left[|\hat\eta(X) - \eta(X)|\right] \leq \left(2\, \mathbb{E}\left[\text{KL}(\eta(X), \hat\eta(X))\right]\right)^{1/2}.$$

**Question** For each method and for $n \in \{50, 200, 1000, 5000\}$ (with $3$ repetitions), compute the excess risk and the two bounds on `test`. Does the bound hold? Is it tight? Which methods have an excess risk that does not vanish when $n$ grows, and why? (The probabilities are clipped in $[10^{-3}, 1-10^{-3}]$ for both bounds, which is infinite when $\hat\eta \in \{0,1\}$.)

**Answer**

In [ ]:
plugin_metrics <- function(fitted, test) {
  eta_hat <- predict(fitted, test, type = "prob")$.pred_Class1
  pred <- if_else(eta_hat > 0.5, "Class1", "Class2")
  eta_clipped <- pmin(pmax(eta_hat, 1e-3), 1 - 1e-3)
  kl <- test$eta * log(test$eta / eta_clipped) + (1 - test$eta) * log((1 - test$eta) / (1 - eta_clipped))
  tibble(`excess risk` = excess_risk(pred, test), `L1 bound` = 2 * mean(abs(eta_clipped - test$eta)),
         `KL bound` = sqrt(2 * mean(kl)))
}
set.seed(1)
bounds <- expand_grid(n = c(50, 200, 1000, 5000), rep = 1:3) |>
  mutate(data = map(n, simulate)) |>
  cross_join(specifications) |>
  mutate(metrics = pmap(list(formula, spec, data), \(formula, spec, data) plugin_metrics(fit_method(formula, spec, data), test))) |>
  select(n, rep, method, metrics) |>
  unnest(metrics)
bounds |>
  group_by(n, method) |> summarise(across(c(`excess risk`, `L1 bound`, `KL bound`), mean), .groups = "drop") |>
  pivot_longer(c(`excess risk`, `L1 bound`, `KL bound`), names_to = "quantity") |>
  ggplot(aes(n, value, color = quantity)) + geom_line() + geom_point() +
  facet_wrap(~ method) + scale_x_log10() + scale_y_log10()

In [ ]:
bounds |> filter(n %in% c(200, 5000)) |>
  group_by(method, n) |> summarise(across(c(`excess risk`, `L1 bound`), mean), .groups = "drop") |>
  pivot_wider(names_from = n, values_from = c(`excess risk`, `L1 bound`))

**Conclusion** The bound holds for all methods and all sizes, but it is far from tight: for the QDA with $n = 200$, the excess risk is $0.004$ while $2\,\mathbb{E}|\hat\eta - \eta| \approx 0.08$. Errors on $\hat\eta$ only matter near the boundary, where they change the decision, while the bound counts them everywhere. The KL bound is even larger. Two groups of methods appear. For the correctly specified ones (QDA, quadratic logistic regression), the excess risk vanishes as $n$ grows ($0.0002$ and $0.0004$ for $n = 5000$): only the *estimation* error remains, and it decreases with $n$. For the misspecified ones (logistic, LDA, naive Bayes), the excess risk stays around $0.04$–$0.05$ whatever $n$: this is the *approximation* error, which data cannot remove. The $k$-NN with a fixed $k = 25$ improves with $n$ ($0.023$ to $0.010$) without reaching $0$: the neighborhood shrinks as $25/n \to 0$, but averaging only 25 labels leaves sampling variance.

---

## 3. Empirical risk minimization with convex surrogates

In the optimization point of view, we look for $f(x) = \beta^\top \phi(x)$ minimizing the empirical risk of a loss $\ell(Y f(X))$ (with labels $Y \in \{-1, +1\}$, $+1$ for `Class1`). The $0/1$ loss $\mathbf{1}_{Yf(X) \leq 0}$ is replaced by a convex *surrogate*. We minimize it directly with `optim()` (with a very small ridge penalty to avoid divergence on separable data).

In [ ]:
phi_linear <- function(d) cbind(1, d$x1, d$x2)
phi_quadratic <- function(d) cbind(1, d$x1, d$x2, d$x1^2, d$x2^2, d$x1 * d$x2)
losses <- list(logistic = \(m) log1p(exp(-m)), exponential = \(m) exp(-m),
               hinge = \(m) pmax(0, 1 - m), squared = \(m) (1 - m)^2)

**Question** Write a function `erm(loss, phi, data, lambda = 1e-4)` returning the coefficients $\hat\beta$, and compute, for the four losses and both feature maps, the excess risk of $\text{sign}(\hat f)$ on `test` (with `train`).

**Answer**

In [ ]:
erm <- function(loss, phi, data, lambda = 1e-4) {
  Phi <- phi(data)
  y <- if_else(data$class == "Class1", 1, -1)
  objective <- function(beta) mean(loss(y * drop(Phi %*% beta))) + lambda * sum(beta[-1]^2)
  optim(rep(0, ncol(Phi)), objective, method = "BFGS", control = list(maxit = 2000))$par
}
erm_fits <- expand_grid(loss = names(losses), features = c("linear", "quadratic")) |>
  mutate(phi = map(features, \(f) if (f == "linear") phi_linear else phi_quadratic),
         beta = map2(loss, phi, \(l, phi) erm(losses[[l]], phi, train)),
         f_test = map2(phi, beta, \(phi, beta) drop(phi(test) %*% beta)),
         `excess risk` = map_dbl(f_test, \(f) excess_risk(if_else(f > 0, "Class1", "Class2"), test)))
erm_fits |> select(loss, features, `excess risk`) |> pivot_wider(names_from = features, values_from = `excess risk`)

**Conclusion** With linear features, all the losses give an excess risk around $0.042$–$0.047$: the approximation error of the linear classifiers dominates, whatever the surrogate. With quadratic features, the excess risks drop to $0.003$ (logistic, exponential), $0.006$ (hinge) and $0.014$ (squared). All the surrogates lead to good classifiers, in this experiment. PC2 proves sign agreement for unrestricted population minimizers where they exist. A restricted feature map can change the optimum. The squared loss is the least efficient here, as it also penalizes the observations that are classified *too* confidently ($Yf > 1$).

---

**Question** PC2 (Exercise 2) shows that the minimizer of the surrogate risk has the sign of $2\eta - 1$, and computes the minimizer $\tilde f(\eta)$ for each loss: $\log\frac{\eta}{1-\eta}$ (logistic), $\frac12\log\frac{\eta}{1-\eta}$ (exponential), $\text{sign}(2\eta - 1)$ (hinge) and $2\eta - 1$ (squared). With the quadratic features, plot $\hat f(x)$ against $\eta(x)$ on $2000$ test points, together with the curves $\tilde f(\eta)$. Which losses give an estimate of $\eta$? Check also that the logistic ERM (with `lambda = 0`) gives the same coefficients as the maximum likelihood fit of `logistic_reg()`.

**Answer**

In [ ]:
f_star <- list(logistic = \(e) qlogis(e), exponential = \(e) qlogis(e) / 2, hinge = \(e) sign(2 * e - 1), squared = \(e) 2 * e - 1)
erm_fits |> filter(features == "quadratic") |>
  mutate(points = map(f_test, \(f) tibble(eta = test$eta[1:2000], f_hat = f[1:2000]))) |>
  select(loss, points) |> unnest(points) |>
  ggplot(aes(eta, f_hat)) + geom_point(size = 0.3, alpha = 0.3) +
  facet_wrap(~ loss, scales = "free_y") +
  geom_line(data = map_dfr(names(f_star), \(l) tibble(loss = l, eta = seq(0.01, 0.99, by = 0.01),
                                                      f_hat = f_star[[l]](seq(0.01, 0.99, by = 0.01)))),
            color = "#eb6834", linewidth = 1)

In [ ]:
mle <- fit(workflow(class ~ x1 + x2 + I(x1^2) + I(x2^2) + I(x1 * x2), logistic_reg()), train) |> tidy()
# glm models P(second level) = P(Class2): its coefficients are the opposite of ours (Class1 = +1)
tibble(term = mle$term, `logistic_reg()` = -mle$estimate, ERM = erm(losses$logistic, phi_quadratic, train, lambda = 0))

**Conclusion** The points follow the predicted curves $\tilde f(\eta)$: the logistic and exponential scores are increasing functions of $\eta$, from which $\eta$ can be recovered ($\eta = 1/(1+e^{-\hat f})$ and $1/(1+e^{-2\hat f})$). The squared loss also gives an estimate, $(1 + \hat f)/2$, but a poor one. The hinge scores saturate around $\pm 1$ away from the boundary: they give the decision, not the probability, as for an SVM. The logistic ERM and the maximum likelihood fit of `logistic_reg()` give the same coefficients (up to the sign convention of the classes): the logistic regression is at the same time a plug-in method (it estimates $\eta$) and an empirical risk minimization of the logistic surrogate.

---

**Question** The $0/1$ loss is not minimized directly. Starting from the logistic solution with the linear features, plot the empirical $0/1$ risk and the empirical logistic risk along the intercept $\beta_0 + t$, for $t \in [-3, 3]$. Why is the $0/1$ empirical risk hard to minimize?

**Answer**

In [ ]:
beta <- erm(losses$logistic, phi_linear, train)
y_train <- if_else(train$class == "Class1", 1, -1)
Phi <- phi_linear(train)
tibble(t = seq(-3, 3, length.out = 601)) |>
  mutate(margins = map(t, \(t) y_train * drop(Phi %*% (beta + c(t, 0, 0)))),
         `0/1 risk` = map_dbl(margins, \(m) mean(m <= 0)),
         `logistic risk` = map_dbl(margins, \(m) mean(log1p(exp(-m))))) |>
  pivot_longer(c(`0/1 risk`, `logistic risk`), names_to = "risk") |>
  ggplot(aes(t, value)) + geom_line() + facet_wrap(~ risk, scales = "free_y")

**Conclusion** The empirical $0/1$ risk is piecewise constant: its gradient is zero almost everywhere, it jumps each time the boundary crosses an observation, and it has many local minima. In dimension $d$, minimizing it is NP-hard in general. The logistic risk is smooth and convex, with a unique minimum: it can be minimized by gradient methods, and its minimizer has the right sign (PC2). This is the *convexification* of the lecture.

---

## 4. Local averaging and the bias–variance trade-off

The $k$-nearest neighbors estimate $\eta(x)$ by the proportion of `Class1` among the $k$ closest training points. With the known $\eta$, we can measure by Monte Carlo the *bias* $\mathbb{E}[\hat\eta(x)] - \eta(x)$ and the *variance* $\text{Var}[\hat\eta(x)]$ of the estimate at some points $x$, over many training sets.

**Question** *Predict before you run:* how should the squared bias and the variance depend on $k$? For $k \in \{1, 5, 15, 45, 101\}$, draw $50$ training sets of size $200$ and compute, averaged over $300$ evaluation points, the squared bias and the variance of $\hat\eta$, as well as the average excess risk (on the first $5000$ test points).

**Answer**

In [ ]:
ks <- c(1, 5, 15, 45, 101)
set.seed(2)
points <- simulate(300)
points <- points |> mutate(eta = eta(points))
test_small <- test |> slice_head(n = 5000)
monte_carlo <- map_dfr(1:50, function(r) {
  data <- simulate(200)
  map_dfr(ks, function(k) {
    fitted_knn <- fit_method(class ~ x1 + x2, nearest_neighbor(neighbors = k) |> set_mode("classification"), data)
    tibble(rep = r, k = k, point = seq_len(nrow(points)), eta = points$eta,
           eta_hat = predict(fitted_knn, points, type = "prob")$.pred_Class1,
           excess = excess_risk(predict(fitted_knn, test_small)$.pred_class, test_small))
  })
})
bias_variance <- monte_carlo |>
  group_by(k, point) |> summarise(bias2 = (mean(eta_hat) - first(eta))^2, variance = var(eta_hat),
                                  excess = mean(excess), .groups = "drop") |>
  group_by(k) |> summarise(`squared bias` = mean(bias2), variance = mean(variance), `excess risk` = mean(excess))
bias_variance
bias_variance |> pivot_longer(-k) |> ggplot(aes(k, value, color = name)) + geom_line() + geom_point() + scale_x_log10()

**Conclusion** The variance of $\hat\eta$ decreases quickly with $k$ (from $0.13$ for $k = 1$ to $0.003$ for $k = 101$, roughly like $1/k$), while the squared bias increases (from $0.003$ to $0.027$): averaging over more neighbors means averaging over a larger region, where $\eta$ varies. The excess risk is U-shaped, with a minimum around $k = 45$ ($0.018$), much better than $k = 1$ ($0.066$, too much variance) or $k = 101$ ($0.028$, too much bias): this is the bias–variance trade-off of the lecture.

---

In practice $\eta$ is unknown and $k$ is chosen by cross validation. With `tidymodels`, a parameter to tune is marked with `tune()`, and `tune_grid()` evaluates a grid of values on resamples (`rsample`).

**Question** On one training set of size $200$, choose $k$ with a $10$-fold cross validation of the accuracy (`tune_grid()`), and compare it with the best $k$ of the Monte Carlo experiment. Then add $8$ useless noise features (`simulate(200, noise = 8)`): what happens to the excess risk of the $25$-nearest neighbors?

**Answer**

In [ ]:
set.seed(3)
data <- simulate(200)
knn_tuning <- tune_grid(workflow(class ~ x1 + x2, nearest_neighbor(neighbors = tune()) |> set_mode("classification")),
                        resamples = vfold_cv(data, v = 10, strata = class), grid = tibble(neighbors = ks),
                        metrics = metric_set(accuracy))
show_best(knn_tuning, metric = "accuracy")

In [ ]:
set.seed(4)
noisy_test <- simulate(5000, noise = 8)
noisy_test <- noisy_test |> mutate(eta = eta(noisy_test))
map_dfr(c(0, 2, 8), function(noise) {
  map_dfr(1:10, \(r) tibble(noise = noise, excess = excess_risk(
    predict(fit_method(class ~ ., nearest_neighbor(neighbors = 25) |> set_mode("classification"), simulate(200, noise = noise)),
            noisy_test |> select(x1, x2, num_range("noise", seq_len(noise))))$.pred_class, noisy_test)))
}) |> group_by(noise) |> summarise(`excess risk of the 25-NN` = mean(excess))

**Conclusion** The cross validation selects $k = 45$, the best value of the Monte Carlo experiment, although its estimates are noisy (standard errors of about $0.02$ on the accuracy, larger than some of the differences between the values of $k$). With useless noise features, the neighbors are computed in a space where most directions carry no information, and the excess risk of the $25$-NN grows from $0.019$ to $0.043$ (2 noise features) and $0.071$ (8 noise features). Local methods suffer from the dimension: this is a first glimpse of the *curse of dimensionality* (Lecture 4).

---

## 5. The tidymodels workflow on real data: TwoClass

We now use the TwoClass data of the lecture (Kuhn and Johnson, *Applied Predictive Modeling*): $208$ observations, two predictors and two classes. The truth is unknown: we have to rely on resampling. A complete `tidymodels` analysis has the following steps:

- `rsample`: split the data into a training and a test set (`initial_split`), and resample the training set (`vfold_cv`),
- `recipes`: describe the preprocessing (normalization, feature design...), estimated on the training part of each resample,
- `parsnip`: describe the models, `workflows`: bundle a recipe and a model,
- `tune` and `workflowsets`: evaluate many workflows and their hyperparameters on the resamples,
- `yardstick`: compute the metrics.

In [ ]:
two_class <- read_csv("data/twoClassData.csv", show_col_types = FALSE) |> mutate(classes = factor(classes))
set.seed(42)
split <- initial_split(two_class, prop = 0.75, strata = classes)
train_tc <- training(split)
test_tc <- testing(split)
folds <- vfold_cv(train_tc, v = 5, repeats = 3, strata = classes)
ggplot(train_tc, aes(PredictorA, PredictorB, color = classes)) + geom_point() + coord_equal()

**Question** Define two recipes (normalized predictors, and normalized predictors with polynomials of degree $2$) and the following models: logistic regression, $k$-NN ($k$ tuned), LDA, QDA, naive Bayes, penalized logistic regression (`glmnet`, penalty and mixture tuned), SVM with a radial kernel (`kernlab`, cost and scale tuned) and a classification tree (cost-complexity tuned). (The perceptron, much slower to tune, is added in *To go further*.) Build a `workflow_set()` crossing them (the polynomial recipe only for the logistic and penalized logistic regressions), and evaluate it with `workflow_map("tune_grid", grid = 6)` on `folds` for the accuracy, the ROC AUC and the Brier score. Rank the workflows.

**Answer**

In [ ]:
mirai::daemons(min(4, parallel::detectCores()))  # tune 2.x runs the resamples in parallel with mirai (2 workers on Colab)
base_recipe <- recipe(classes ~ ., data = train_tc) |> step_normalize(all_predictors())
poly_recipe <- base_recipe |> step_poly(all_predictors(), degree = 2)
models <- list(
  logistic = logistic_reg(),
  knn = nearest_neighbor(neighbors = tune()) |> set_mode("classification"),
  lda = discrim_linear(),
  qda = discrim_quad(),
  naive_bayes = naive_Bayes() |> set_engine("naivebayes"),
  glmnet = logistic_reg(penalty = tune(), mixture = tune()) |> set_engine("glmnet"),
  svm = svm_rbf(cost = tune(), rbf_sigma = tune()) |> set_mode("classification"),
  tree = decision_tree(cost_complexity = tune()) |> set_mode("classification"))
workflows <- bind_rows(
  workflow_set(preproc = list(base = base_recipe), models = models),
  workflow_set(preproc = list(poly = poly_recipe), models = models[c("logistic", "glmnet")]))
results <- workflows |>
  workflow_map("tune_grid", resamples = folds, grid = 6, seed = 42,
               metrics = metric_set(accuracy, roc_auc, brier_class),
               control = control_grid(save_pred = TRUE, save_workflow = TRUE))
rank_results(results, rank_metric = "roc_auc", select_best = TRUE) |>
  select(wflow_id, .metric, mean, std_err, rank) |>
  pivot_wider(names_from = .metric, values_from = c(mean, std_err))

In [ ]:
autoplot(results, rank_metric = "roc_auc", metric = c("accuracy", "roc_auc"), select_best = TRUE)

**Conclusion** With the ROC AUC, the tuned $k$-NN comes first ($0.83$), followed by the naive Bayes ($0.82$), the logistic and penalized logistic regressions with quadratic features ($0.81$) and the SVM ($0.80$). The linear methods (logistic, LDA, penalized logistic) and the tree are around $0.79$, and the QDA comes last ($0.78$). The rankings by accuracy and by AUC differ (the quadratic logistic regressions have the best accuracies, $0.78$). All the differences are of the order of the standard errors ($0.01$–$0.02$): with $156$ training observations, many methods are statistically indistinguishable.

---

**Question** The differences between the best workflows are small. Compare the two best ones with their *paired* differences of ROC AUC on the $15$ resamples (their best configurations are evaluated on the same folds). Then finalize the best workflow, fit it on the whole training set and evaluate it once on the test set with `last_fit()`. Plot its decision map.

**Answer**

In [ ]:
best_two <- rank_results(results, rank_metric = "roc_auc", select_best = TRUE) |>
  filter(.metric == "roc_auc") |> slice_head(n = 2) |> pull(wflow_id)
per_fold <- map_dfr(best_two, function(id) {
  result <- extract_workflow_set_result(results, id)
  collect_metrics(result, summarize = FALSE) |>
    filter(.metric == "roc_auc", .config == select_best(result, metric = "roc_auc")$.config) |>
    transmute(id, id2, wflow_id = !!id, .estimate)
})
differences <- per_fold |> pivot_wider(names_from = wflow_id, values_from = .estimate) |>
  mutate(difference = .data[[best_two[1]]] - .data[[best_two[2]]])
c(mean = mean(differences$difference), sd = sd(differences$difference),
  `positive differences` = sum(differences$difference > 0), resamples = nrow(differences))

In [ ]:
best_result <- extract_workflow_set_result(results, best_two[1])
final <- extract_workflow(results, best_two[1]) |>
  finalize_workflow(select_best(best_result, metric = "roc_auc")) |>
  last_fit(split, metrics = metric_set(accuracy, roc_auc, brier_class))
collect_metrics(final)

In [ ]:
final_fit <- extract_workflow(final)
grid_tc <- expand_grid(PredictorA = seq(0, 0.8, length.out = 150), PredictorB = seq(0, 0.7, length.out = 150))
grid_tc |> bind_cols(predict(final_fit, grid_tc, type = "prob")) |>
  ggplot(aes(PredictorA, PredictorB)) + geom_raster(aes(fill = .pred_Class1), alpha = 0.6) +
  geom_contour(aes(z = .pred_Class1), breaks = 0.5, color = "black") +
  geom_point(data = two_class, aes(color = classes)) +
  scale_fill_gradient2(low = "#2a78d6", mid = "white", high = "#eb6834", midpoint = 0.5) +
  scale_color_manual(values = c(Class1 = "#eb6834", Class2 = "#2a78d6")) + coord_equal(expand = FALSE)

**Conclusion** The paired differences of AUC between the two best workflows ($k$-NN and naive Bayes) have a mean of $0.008$ and a standard deviation of $0.037$, and only $9$ of the $15$ are positive: the two are indistinguishable. The test set gives an accuracy of $0.72$ and an AUC of $0.76$ for the $k$-NN, *below* its CV estimate ($0.83$). Part of the gap is the optimism of the selection (the best of $10$ workflows tuned on the same folds, see Lab 1), and part is the variability of a test set of only $52$ observations (the standard error of an accuracy is then about $0.06$).

---

**Question** Which phenomena of parts 2 to 4 (misspecification, flexibility, bias–variance trade-off) explain the ranking on TwoClass?

**Answer** The boundary of TwoClass is curved, so the linear methods (logistic, LDA, linear penalized logistic) are misspecified: as on the simulated problem, they keep an approximation error and lose against the flexible methods (the tuned $k$-NN, and the naive Bayes, whose class densities give a curved boundary) and against the quadratic features. The QDA has the right *form* of boundary but estimates two covariance matrices with few observations, and its Gaussian assumption is wrong for these skewed predictors: it pays in variance and bias. The tuned $k$-NN wins by choosing its bias–variance trade-off with the cross validation, but with $156$ observations the estimation error and the variability of the estimates (Part 4) are large enough to blur the ranking: several methods are equivalent, and the simplest one, the logistic regression with quadratic features, is a sensible choice.

---

## About this lab

**Use of AI.** Artificial intelligence tools have been used to revise the content of this lab. The main contributor takes full responsibility for this content.

**License.** The texts of this lab are under the [Creative Commons Attribution-ShareAlike 4.0](https://creativecommons.org/licenses/by-sa/4.0/) license (CC BY-SA 4.0), and its code under the [MIT](https://opensource.org/license/mit) license; the datasets keep their own licenses. Main contributor: E. Le Pennec. Contributors: S. Boucheron, A. Dieuleveut, A.K. Fermin, S. Gadat, S. Gaiffas, B. Goujaud, A. Guilloux, Ch. Keribin, Th. Klein, E. Matzner, J. Read, M. Sangnier, E. Scornet.